# FieldVision — Stage 5.1
## SLA Refinement and Controlled Simulator Validation

This notebook refines the Stage 5 development simulator before reinforcement learning.

It makes two key changes:

1. **Service criticality and deadline urgency are independent.**
   - Criticality answers: *How important is this service?*
   - Deadline urgency answers: *How soon must this task finish?*

2. It adds controlled validation tests for:
   - weighted wireless sharing,
   - edge-compute queueing,
   - temporary network outages,
   - burst arrivals,
   - EDF versus priority-aware scheduling.

The goal is to validate the scientific simulator mechanics before MAPPO is introduced.

## 0. Imports and paths

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HOME = Path.home()
PROJECT_ROOT = HOME / "Downloads" / "NOMS_Fieldvision"
SIM_DIR = PROJECT_ROOT / "simulator_inputs"
RESULTS_DIR = PROJECT_ROOT / "stage5_1_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRACE_PATH = SIM_DIR / "fieldvision_task_trace.csv"

if not TRACE_PATH.exists():
    raise FileNotFoundError(
        f"{TRACE_PATH} not found. Run Stage 4 first."
    )

base_trace = pd.read_csv(TRACE_PATH)

print("Loaded tasks:", len(base_trace))
display(base_trace.head())

Loaded tasks: 5000


,task_id,arrival_time_s,uav_id,workload,service_class,priority_weight,experimental_deadline_s,input_size_mb,megapixels,measured_decode_ms,measured_preprocess_ms,measured_compute_ms_current_host,measured_output_kb,compute_profile_device,compute_profile_model,source_dataset,reference_upload_s_at_25mbps,reference_completion_s,compute_demand_normalized,network_demand_normalized
0,task_000000,2.404209,1,Leafy context classification,delay_tolerant,1,6.697156,4.000870,1.048576,5.536792,3.498250,50.117792,3.90625,cpu,MobileNetV3-Small,LeafySpurge,1.280278,1.339431,0.480804,1.000000
1,task_000001,4.740398,2,Leafy orthomosaic tiled analysis,critical,3,17.477765,34.676382,16.777216,75.166375,8.579459,2802.024183,15.62500,cpu,MobileNetV3-Small tiled proxy,LeafySpurge,11.096442,13.982212,26.881185,8.667211
2,task_000002,7.125159,2,Leafy orthomosaic tiled analysis,delay_tolerant,1,54.395912,25.624428,16.777216,328.896334,7.505665,2342.963567,15.62500,cpu,MobileNetV3-Small tiled proxy,LeafySpurge,8.199817,10.879182,22.477193,6.404714
3,task_000003,7.404954,3,Leafy crop classification,standard,2,0.102975,0.006172,0.001521,1.633709,1.045125,46.833334,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.051487,0.449295,0.001543
4,task_000004,7.491391,1,Leafy crop classification,critical,3,0.065461,0.006172,0.001521,1.795750,0.873667,47.724042,3.90625,cpu,MobileNetV3-Small,LeafySpurge,0.001975,0.052369,0.457840,0.001543


## 1. Refine the task semantics

Stage 4 tied priority and deadline together. Here they are separated.

**Criticality** controls service importance / resource preference.

**Urgency** controls how tight the deadline is.

These are still experimental systems classes, not agronomic requirements.

In [2]:
SEED = 42
rng = np.random.default_rng(SEED)

CRITICALITY = {
    "high":   {"probability": 0.20, "priority_weight": 3},
    "normal": {"probability": 0.60, "priority_weight": 2},
    "low":    {"probability": 0.20, "priority_weight": 1},
}

URGENCY = {
    "urgent":  {"probability": 0.25, "deadline_multiplier": 1.25},
    "normal":  {"probability": 0.55, "deadline_multiplier": 2.00},
    "relaxed": {"probability": 0.20, "deadline_multiplier": 4.00},
}

REFERENCE_BANDWIDTH_MBPS = 25.0

def draw_category(spec, rng):
    names = list(spec.keys())
    probs = np.array([spec[n]["probability"] for n in names], dtype=float)
    probs = probs / probs.sum()
    return str(rng.choice(names, p=probs))

trace = base_trace.copy()

criticality = []
urgency = []
priority = []
deadlines = []

for _, row in trace.iterrows():
    c = draw_category(CRITICALITY, rng)
    u = draw_category(URGENCY, rng)

    # Workload-specific reference completion scale.
    # This is derived from empirical input size + Stage 3 timing,
    # but the resulting deadline remains an experimental assumption.
    upload_s = float(row["input_size_mb"]) * 8.0 / REFERENCE_BANDWIDTH_MBPS

    compute_s = (
        float(row["measured_decode_ms"])
        + float(row["measured_preprocess_ms"])
        + float(row["measured_compute_ms_current_host"])
    ) / 1000.0

    reference_s = upload_s + compute_s

    criticality.append(c)
    urgency.append(u)
    priority.append(CRITICALITY[c]["priority_weight"])
    deadlines.append(reference_s * URGENCY[u]["deadline_multiplier"])

trace["criticality"] = criticality
trace["urgency"] = urgency
trace["priority_weight"] = priority
trace["experimental_deadline_s_v2"] = deadlines

print("Criticality distribution:")
display(trace["criticality"].value_counts(normalize=True).rename("fraction"))

print("Urgency distribution:")
display(trace["urgency"].value_counts(normalize=True).rename("fraction"))

print("Cross-tab: criticality vs urgency")
display(pd.crosstab(trace["criticality"], trace["urgency"]))

Criticality distribution:


criticality
normal    0.6068
high      0.2016
low       0.1916
Name: fraction, dtype: float64

Urgency distribution:


urgency
normal     0.5420
urgent     0.2588
relaxed    0.1992
Name: fraction, dtype: float64

Cross-tab: criticality vs urgency


urgency,normal,relaxed,urgent
criticality,,,
high,538,206,264
low,493,209,256
normal,1679,581,774


## 2. Check that criticality and urgency are no longer the same signal

In [3]:
cross = pd.crosstab(
    trace["criticality"],
    trace["urgency"],
    normalize="index"
)

display(cross)

print(
    "High-criticality tasks that are NOT urgent:",
    (
        (trace["criticality"] == "high")
        & (trace["urgency"] != "urgent")
    ).sum()
)

print(
    "Urgent tasks that are NOT high-criticality:",
    (
        (trace["urgency"] == "urgent")
        & (trace["criticality"] != "high")
    ).sum()
)

urgency,normal,relaxed,urgent
criticality,,,
high,0.533730,0.204365,0.261905
low,0.514614,0.218163,0.267223
normal,0.553395,0.191496,0.255109


High-criticality tasks that are NOT urgent: 744
Urgent tasks that are NOT high-criticality: 1030


## 3. Refined Stage 5 simulator

This is the same basic architecture as Stage 5, but it uses:
- `criticality` for priority-aware scheduling and network tiering;
- `experimental_deadline_s_v2` for SLA timing.

In [4]:
DEV = {
    "dt_s": 0.02,
    "arrival_horizon_s": 180.0,
    "drain_s": 90.0,
    "top_k_tasks": 3,

    # Development-only multipliers until real UAV/edge/cloud profiles exist.
    "compute_multiplier": {
        "local": 2.0,
        "edge": 0.75,
        "cloud": 0.50,
    },

    "edge_slots": 2,
    "cloud_slots": 4,

    "fixed_remote_delay_s": {
        "edge": 0.020,
        "cloud": 0.080,
    },

    "network_tier_weight": {
        "low": 1.0,
        "normal": 2.0,
        "high": 4.0,
    },

    "network": {
        "seed": 123,
        "initial_mbps": 25.0,
        "min_mbps": 3.0,
        "max_mbps": 50.0,
        "update_interval_s": 1.0,
        "lognormal_sigma": 0.25,
        "outage_probability_per_update": 0.01,
        "outage_min_s": 1.0,
        "outage_max_s": 3.0,
        "outage_capacity_mbps": 0.2,
    },
}


class DevelopmentNetworkModel:
    def __init__(self, cfg):
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg["seed"])
        self.current_mbps = float(cfg["initial_mbps"])
        self.next_update_s = 0.0
        self.outage_until_s = -1.0

    def capacity_mbps(self, now_s):
        if now_s < self.outage_until_s:
            return float(self.cfg["outage_capacity_mbps"])

        if now_s >= self.next_update_s:
            self.next_update_s = now_s + self.cfg["update_interval_s"]

            if self.rng.random() < self.cfg["outage_probability_per_update"]:
                duration = self.rng.uniform(
                    self.cfg["outage_min_s"],
                    self.cfg["outage_max_s"]
                )
                self.outage_until_s = now_s + duration
                return float(self.cfg["outage_capacity_mbps"])

            factor = self.rng.lognormal(
                mean=0.0,
                sigma=self.cfg["lognormal_sigma"]
            )

            proposed = (
                0.70 * self.current_mbps
                + 0.30 * self.cfg["initial_mbps"] * factor
            )

            self.current_mbps = float(np.clip(
                proposed,
                self.cfg["min_mbps"],
                self.cfg["max_mbps"]
            ))

        return self.current_mbps


class FieldVisionStage51Sim:
    def __init__(self, full_trace_df, n_uavs=4, dev_cfg=DEV, policy_seed=7):
        self.cfg = dev_cfg
        self.dt = float(dev_cfg["dt_s"])
        self.n_uavs = int(n_uavs)
        self.policy_rng = np.random.default_rng(policy_seed)

        self.trace = (
            full_trace_df[
                full_trace_df["arrival_time_s"]
                < dev_cfg["arrival_horizon_s"]
            ]
            .sort_values("arrival_time_s")
            .reset_index(drop=True)
            .copy()
        )

        self.end_time_s = (
            dev_cfg["arrival_horizon_s"]
            + dev_cfg["drain_s"]
        )

        self.reset()

    def reset(self):
        self.now_s = 0.0
        self.arrival_idx = 0

        self.tasks = {}
        self.waiting = {u: [] for u in range(self.n_uavs)}

        self.local_active = {u: None for u in range(self.n_uavs)}
        self.transfers = {}

        self.edge_queue = []
        self.cloud_queue = []
        self.edge_active = []
        self.cloud_active = []

        self.completed_task_ids = []

        self.total_network_capacity_mb = 0.0
        self.total_network_transferred_mb = 0.0
        self.edge_busy_slot_seconds = 0.0
        self.cloud_busy_slot_seconds = 0.0
        self.local_busy_slot_seconds = 0.0

        self.system_log = []

        self.network = DevelopmentNetworkModel(self.cfg["network"])

    def _task_processing_time_s(self, task, target):
        base_s = (
            float(task["measured_decode_ms"])
            + float(task["measured_preprocess_ms"])
            + float(task["measured_compute_ms_current_host"])
        ) / 1000.0

        return base_s * self.cfg["compute_multiplier"][target]

    def _absolute_deadline(self, task):
        return (
            float(task["arrival_time_s"])
            + float(task["experimental_deadline_s_v2"])
        )

    def _inject_arrivals(self):
        while self.arrival_idx < len(self.trace):
            row = self.trace.iloc[self.arrival_idx]

            if float(row["arrival_time_s"]) > self.now_s + 1e-12:
                break

            task = row.to_dict()

            task_id = str(task["task_id"])
            uav = int(task["uav_id"])

            task["absolute_deadline_s"] = self._absolute_deadline(task)
            task["status"] = "waiting"
            task["selected_target"] = None
            task["network_tier"] = None
            task["completion_time_s"] = None
            task["deadline_missed"] = False

            self.tasks[task_id] = task
            self.waiting[uav].append(task_id)

            self.arrival_idx += 1

    def waiting_candidates(self, uav_id, top_k=None):
        if top_k is None:
            top_k = self.cfg["top_k_tasks"]

        ids = list(self.waiting[uav_id])

        ids.sort(
            key=lambda tid: (
                self.tasks[tid]["absolute_deadline_s"],
                -float(self.tasks[tid]["priority_weight"])
            )
        )

        return [self.tasks[tid] for tid in ids[:top_k]]

    def uav_has_active_transfer(self, uav_id):
        return any(
            tr["uav_id"] == uav_id
            for tr in self.transfers.values()
        )

    def current_network_mbps(self):
        return self.network.capacity_mbps(self.now_s)

    def estimated_completion_from_now_s(self, task, target):
        process_s = self._task_processing_time_s(task, target)

        if target == "local":
            if self.local_active[int(task["uav_id"])] is not None:
                return np.inf
            return process_s

        if self.uav_has_active_transfer(int(task["uav_id"])):
            return np.inf

        bw = max(self.current_network_mbps(), 1e-6)
        upload_s = float(task["input_size_mb"]) * 8.0 / bw

        if target == "edge":
            q = len(self.edge_queue) + len(self.edge_active)
            queue_penalty = (q / max(self.cfg["edge_slots"], 1)) * process_s
        elif target == "cloud":
            q = len(self.cloud_queue) + len(self.cloud_active)
            queue_penalty = (q / max(self.cfg["cloud_slots"], 1)) * process_s
        else:
            return np.inf

        return (
            upload_s
            + self.cfg["fixed_remote_delay_s"][target]
            + queue_penalty
            + process_s
        )

    def apply_actions(self, actions):
        for uav_id, action in actions.items():
            if action is None:
                continue

            target = action.get("target", "defer")

            if target == "defer":
                continue

            task_id = action.get("task_id")

            if task_id not in self.waiting[uav_id]:
                continue

            task = self.tasks[task_id]

            if target == "local":
                if self.local_active[uav_id] is not None:
                    continue

                self.waiting[uav_id].remove(task_id)

                task["status"] = "running_local"
                task["selected_target"] = "local"

                self.local_active[uav_id] = {
                    "task_id": task_id,
                    "remaining_s": self._task_processing_time_s(task, "local"),
                }

            elif target in {"edge", "cloud"}:
                if self.uav_has_active_transfer(uav_id):
                    continue

                tier = action.get("network_tier", "normal")
                if tier not in self.cfg["network_tier_weight"]:
                    tier = "normal"

                self.waiting[uav_id].remove(task_id)

                task["status"] = "transmitting"
                task["selected_target"] = target
                task["network_tier"] = tier

                self.transfers[task_id] = {
                    "task_id": task_id,
                    "uav_id": uav_id,
                    "target": target,
                    "network_tier": tier,
                    "weight": float(
                        self.cfg["network_tier_weight"][tier]
                    ),
                    "remaining_mb": float(task["input_size_mb"]),
                }

    def _advance_network(self):
        capacity_mbps = self.current_network_mbps()

        self.total_network_capacity_mb += (
            capacity_mbps * self.dt / 8.0
        )

        if not self.transfers:
            return

        total_weight = sum(
            tr["weight"]
            for tr in self.transfers.values()
        )

        finished = []

        for task_id, tr in list(self.transfers.items()):
            share_mbps = (
                capacity_mbps
                * tr["weight"]
                / total_weight
            )

            possible_mb = share_mbps * self.dt / 8.0

            actual_mb = min(
                possible_mb,
                tr["remaining_mb"]
            )

            tr["remaining_mb"] -= actual_mb
            self.total_network_transferred_mb += actual_mb

            if tr["remaining_mb"] <= 1e-12:
                finished.append(task_id)

        for task_id in finished:
            tr = self.transfers.pop(task_id)
            task = self.tasks[task_id]
            target = tr["target"]

            task["status"] = f"queued_{target}"
            task["remote_ready_time_s"] = (
                self.now_s
                + self.cfg["fixed_remote_delay_s"][target]
            )

            if target == "edge":
                self.edge_queue.append(task_id)
            else:
                self.cloud_queue.append(task_id)

    def _sorted_ready_queue(self, queue_ids):
        ready = [
            tid for tid in queue_ids
            if float(
                self.tasks[tid].get("remote_ready_time_s", 0.0)
            ) <= self.now_s
        ]

        # Remote infrastructure prioritizes criticality first,
        # then deadline.
        ready.sort(
            key=lambda tid: (
                -float(self.tasks[tid]["priority_weight"]),
                float(self.tasks[tid]["absolute_deadline_s"]),
            )
        )

        return ready

    def _start_remote_jobs(self, queue_ids, active_jobs, slots, target):
        free_slots = slots - len(active_jobs)

        if free_slots <= 0:
            return

        ready = self._sorted_ready_queue(queue_ids)

        for task_id in ready[:free_slots]:
            queue_ids.remove(task_id)

            task = self.tasks[task_id]
            task["status"] = f"running_{target}"

            active_jobs.append({
                "task_id": task_id,
                "remaining_s": self._task_processing_time_s(task, target),
            })

    def _complete_task(self, task_id, completion_time_s):
        task = self.tasks[task_id]
        task["status"] = "completed"
        task["completion_time_s"] = completion_time_s
        task["deadline_missed"] = (
            completion_time_s > task["absolute_deadline_s"]
        )
        self.completed_task_ids.append(task_id)

    def _advance_compute(self):
        self._start_remote_jobs(
            self.edge_queue,
            self.edge_active,
            self.cfg["edge_slots"],
            "edge"
        )

        self._start_remote_jobs(
            self.cloud_queue,
            self.cloud_active,
            self.cfg["cloud_slots"],
            "cloud"
        )

        self.edge_busy_slot_seconds += len(self.edge_active) * self.dt
        self.cloud_busy_slot_seconds += len(self.cloud_active) * self.dt
        self.local_busy_slot_seconds += (
            sum(job is not None for job in self.local_active.values())
            * self.dt
        )

        completion_time = self.now_s + self.dt

        for uav_id, job in list(self.local_active.items()):
            if job is None:
                continue

            job["remaining_s"] -= self.dt

            if job["remaining_s"] <= 1e-12:
                self._complete_task(job["task_id"], completion_time)
                self.local_active[uav_id] = None

        remaining_edge = []
        for job in self.edge_active:
            job["remaining_s"] -= self.dt
            if job["remaining_s"] <= 1e-12:
                self._complete_task(job["task_id"], completion_time)
            else:
                remaining_edge.append(job)
        self.edge_active = remaining_edge

        remaining_cloud = []
        for job in self.cloud_active:
            job["remaining_s"] -= self.dt
            if job["remaining_s"] <= 1e-12:
                self._complete_task(job["task_id"], completion_time)
            else:
                remaining_cloud.append(job)
        self.cloud_active = remaining_cloud

    def _update_deadline_flags(self):
        next_time = self.now_s + self.dt

        for task in self.tasks.values():
            if task["status"] == "completed":
                continue

            if next_time > task["absolute_deadline_s"]:
                task["deadline_missed"] = True

    def _log_state(self):
        step_idx = int(round(self.now_s / self.dt))
        every = max(1, int(round(1.0 / self.dt)))

        if step_idx % every != 0:
            return

        self.system_log.append({
            "time_s": self.now_s,
            "network_capacity_mbps": self.current_network_mbps(),
            "active_transfers": len(self.transfers),
            "edge_queue": len(self.edge_queue),
            "edge_active": len(self.edge_active),
            "cloud_queue": len(self.cloud_queue),
            "cloud_active": len(self.cloud_active),
            "waiting_total": sum(len(q) for q in self.waiting.values()),
            "completed": len(self.completed_task_ids),
        })

    def step(self, policy_fn):
        self._inject_arrivals()
        actions = policy_fn(self)
        self.apply_actions(actions)
        self._advance_network()
        self._advance_compute()
        self._update_deadline_flags()
        self._log_state()
        self.now_s += self.dt

    def run(self, policy_fn):
        while self.now_s < self.end_time_s:
            self.step(policy_fn)
        return self.metrics()

    def task_results_dataframe(self):
        rows = []

        for task_id, task in self.tasks.items():
            completion = task["completion_time_s"]

            latency = (
                completion - float(task["arrival_time_s"])
                if completion is not None else np.nan
            )

            sla_success = (
                completion is not None
                and completion <= task["absolute_deadline_s"]
            )

            rows.append({
                "task_id": task_id,
                "uav_id": int(task["uav_id"]),
                "workload": task["workload"],
                "criticality": task["criticality"],
                "urgency": task["urgency"],
                "priority_weight": task["priority_weight"],
                "arrival_time_s": task["arrival_time_s"],
                "deadline_s": task["absolute_deadline_s"],
                "target": task["selected_target"],
                "network_tier": task["network_tier"],
                "completion_time_s": completion,
                "latency_s": latency,
                "sla_success": bool(sla_success),
                "deadline_missed": bool(task["deadline_missed"]),
                "input_size_mb": task["input_size_mb"],
            })

        return pd.DataFrame(rows)

    def metrics(self):
        df = self.task_results_dataframe()
        completed = df["completion_time_s"].notna()
        completed_df = df[completed]

        p50 = completed_df["latency_s"].median() if len(completed_df) else np.nan
        p95 = (
            np.percentile(completed_df["latency_s"], 95)
            if len(completed_df) else np.nan
        )

        total_sim_s = self.end_time_s

        return {
            "tasks": len(df),
            "completion_rate": float(completed.mean()),
            "sla_success_rate": float(df["sla_success"].mean()),
            "latency_p50_s": float(p50),
            "latency_p95_s": float(p95),
            "network_utilization": (
                self.total_network_transferred_mb
                / self.total_network_capacity_mb
                if self.total_network_capacity_mb > 0 else 0.0
            ),
            "local_compute_utilization": (
                self.local_busy_slot_seconds
                / (self.n_uavs * total_sim_s)
            ),
            "edge_compute_utilization": (
                self.edge_busy_slot_seconds
                / (self.cfg["edge_slots"] * total_sim_s)
            ),
            "cloud_compute_utilization": (
                self.cloud_busy_slot_seconds
                / (self.cfg["cloud_slots"] * total_sim_s)
            ),
        }

## 4. EDF and priority-aware policies now use different signals

In [5]:
def network_tier_for_criticality(task):
    if task["criticality"] == "high":
        return "high"
    if task["criticality"] == "low":
        return "low"
    return "normal"


def choose_fastest_target(env, task):
    estimates = {
        target: env.estimated_completion_from_now_s(task, target)
        for target in ["local", "edge", "cloud"]
    }
    return min(estimates, key=estimates.get)


def edf_policy(env):
    actions = {}

    for uav in range(env.n_uavs):
        candidates = env.waiting_candidates(uav)
        if not candidates:
            continue

        task = min(
            candidates,
            key=lambda t: t["absolute_deadline_s"]
        )

        target = choose_fastest_target(env, task)

        if not np.isfinite(
            env.estimated_completion_from_now_s(task, target)
        ):
            continue

        actions[uav] = {
            "task_id": task["task_id"],
            "target": target,
            # EDF does not use criticality to select the task,
            # but the service request may still carry a network tier.
            "network_tier": network_tier_for_criticality(task),
        }

    return actions


def priority_aware_policy(env):
    actions = {}

    for uav in range(env.n_uavs):
        candidates = env.waiting_candidates(uav)
        if not candidates:
            continue

        task = sorted(
            candidates,
            key=lambda t: (
                -float(t["priority_weight"]),
                float(t["absolute_deadline_s"]),
            )
        )[0]

        target = choose_fastest_target(env, task)

        if not np.isfinite(
            env.estimated_completion_from_now_s(task, target)
        ):
            continue

        actions[uav] = {
            "task_id": task["task_id"],
            "target": target,
            "network_tier": network_tier_for_criticality(task),
        }

    return actions

# Controlled Validation Tests

These are not performance experiments. They are simulator unit tests.

## Test 1 — Weighted wireless sharing

In [6]:
weights = {
    "low": DEV["network_tier_weight"]["low"],
    "normal": DEV["network_tier_weight"]["normal"],
    "high": DEV["network_tier_weight"]["high"],
}

capacity = 28.0
total_weight = sum(weights.values())

shares = {
    tier: capacity * weight / total_weight
    for tier, weight in weights.items()
}

print("Shared capacity:", capacity, "Mbps")
print("Expected weighted shares:", shares)

assert shares["high"] > shares["normal"] > shares["low"]

ratio = np.array([
    shares["low"],
    shares["normal"],
    shares["high"],
]) / shares["low"]

print("Relative allocation:", ratio)
assert np.allclose(ratio, [1, 2, 4])

print("PASS: low/normal/high receive 1:2:4 weighted shares.")

Shared capacity: 28.0 Mbps
Expected weighted shares: {'low': 4.0, 'normal': 8.0, 'high': 16.0}
Relative allocation: [1. 2. 4.]
PASS: low/normal/high receive 1:2:4 weighted shares.


## Test 2 — Edge queue criticality ordering

In [7]:
# Minimal direct validation of the remote queue ordering rule.

synthetic_tasks = {
    "low_urgent": {
        "priority_weight": 1,
        "absolute_deadline_s": 1.0,
        "remote_ready_time_s": 0.0,
    },
    "high_later": {
        "priority_weight": 3,
        "absolute_deadline_s": 5.0,
        "remote_ready_time_s": 0.0,
    },
    "normal_mid": {
        "priority_weight": 2,
        "absolute_deadline_s": 3.0,
        "remote_ready_time_s": 0.0,
    },
}

ordered = sorted(
    synthetic_tasks.keys(),
    key=lambda tid: (
        -synthetic_tasks[tid]["priority_weight"],
        synthetic_tasks[tid]["absolute_deadline_s"],
    )
)

print("Queue order:", ordered)

assert ordered == [
    "high_later",
    "normal_mid",
    "low_urgent",
]

print("PASS: shared edge queue prioritizes criticality, then deadline.")

Queue order: ['high_later', 'normal_mid', 'low_urgent']
PASS: shared edge queue prioritizes criticality, then deadline.


## Test 3 — Temporary outage slows an in-flight transfer

In [8]:
def transfer_progress_mb(capacity_mbps, duration_s):
    return capacity_mbps * duration_s / 8.0

normal_capacity = 25.0
outage_capacity = 0.2

normal_progress = transfer_progress_mb(normal_capacity, 2.0)
outage_progress = transfer_progress_mb(outage_capacity, 2.0)

print("2 s progress at 25 Mbps:", normal_progress, "MB")
print("2 s progress during outage:", outage_progress, "MB")

assert outage_progress < normal_progress * 0.02

print("PASS: outage nearly pauses transfer progress.")

2 s progress at 25 Mbps: 6.25 MB
2 s progress during outage: 0.05 MB
PASS: outage nearly pauses transfer progress.


## Test 4 — Burst arrivals create and then drain a queue

In [9]:
burst_cfg = dict(DEV)
burst_cfg["arrival_horizon_s"] = 5.0
burst_cfg["drain_s"] = 30.0
burst_cfg["edge_slots"] = 1

# Ten small tasks arriving at the same time to one UAV.
template = trace.iloc[0].copy()

burst_rows = []

for i in range(10):
    row = template.copy()
    row["task_id"] = f"burst_{i}"
    row["arrival_time_s"] = 0.0
    row["uav_id"] = 0
    row["criticality"] = "normal"
    row["urgency"] = "relaxed"
    row["priority_weight"] = 2
    row["experimental_deadline_s_v2"] = 30.0
    row["input_size_mb"] = 0.01
    row["measured_decode_ms"] = 2.0
    row["measured_preprocess_ms"] = 2.0
    row["measured_compute_ms_current_host"] = 250.0
    burst_rows.append(row)

burst_trace = pd.DataFrame(burst_rows)


def burst_local_policy(env):
    actions = {}

    uav = 0
    candidates = env.waiting_candidates(uav)

    if candidates and env.local_active[uav] is None:
        task = candidates[0]
        actions[uav] = {
            "task_id": task["task_id"],
            "target": "local",
            "network_tier": "normal",
        }

    return actions


burst_env = FieldVisionStage51Sim(
    burst_trace,
    n_uavs=1,
    dev_cfg=burst_cfg,
)

burst_env.run(burst_local_policy)

burst_log = pd.DataFrame(burst_env.system_log)

print("Maximum waiting queue:", burst_log["waiting_total"].max())
print("Final waiting queue:", burst_log["waiting_total"].iloc[-1])
print("Completed:", len(burst_env.completed_task_ids))

assert burst_log["waiting_total"].max() > 0
assert len(burst_env.completed_task_ids) == 10

print("PASS: burst creates backlog and the queue later drains.")

Maximum waiting queue: 9
Final waiting queue: 0
Completed: 10
PASS: burst creates backlog and the queue later drains.


## Test 5 — EDF and priority-aware scheduling diverge

In [10]:
# Two tasks at the same UAV:
# - low-criticality task has the earlier deadline
# - high-criticality task has the later deadline

base = trace.iloc[0].copy()

task_a = base.copy()
task_a["task_id"] = "low_urgent"
task_a["arrival_time_s"] = 0.0
task_a["uav_id"] = 0
task_a["criticality"] = "low"
task_a["priority_weight"] = 1
task_a["urgency"] = "urgent"
task_a["experimental_deadline_s_v2"] = 1.0
task_a["input_size_mb"] = 0.01
task_a["measured_compute_ms_current_host"] = 300.0
task_a["measured_decode_ms"] = 1.0
task_a["measured_preprocess_ms"] = 1.0

task_b = base.copy()
task_b["task_id"] = "high_later"
task_b["arrival_time_s"] = 0.0
task_b["uav_id"] = 0
task_b["criticality"] = "high"
task_b["priority_weight"] = 3
task_b["urgency"] = "normal"
task_b["experimental_deadline_s_v2"] = 3.0
task_b["input_size_mb"] = 0.01
task_b["measured_compute_ms_current_host"] = 300.0
task_b["measured_decode_ms"] = 1.0
task_b["measured_preprocess_ms"] = 1.0

two_task_trace = pd.DataFrame([task_a, task_b])

test_env_edf = FieldVisionStage51Sim(
    two_task_trace,
    n_uavs=1,
    dev_cfg={
        **DEV,
        "arrival_horizon_s": 1.0,
        "drain_s": 10.0,
    },
)

test_env_edf._inject_arrivals()
edf_action = edf_policy(test_env_edf)

test_env_prio = FieldVisionStage51Sim(
    two_task_trace,
    n_uavs=1,
    dev_cfg={
        **DEV,
        "arrival_horizon_s": 1.0,
        "drain_s": 10.0,
    },
)

test_env_prio._inject_arrivals()
prio_action = priority_aware_policy(test_env_prio)

print("EDF first choice:", edf_action[0]["task_id"])
print("Priority-aware first choice:", prio_action[0]["task_id"])

assert edf_action[0]["task_id"] == "low_urgent"
assert prio_action[0]["task_id"] == "high_later"

print("PASS: EDF and priority-aware scheduling now make different decisions.")

EDF first choice: low_urgent
Priority-aware first choice: high_later
PASS: EDF and priority-aware scheduling now make different decisions.


## 5. Run the refined policies on the real Stage 4-derived trace

In [11]:
POLICIES = {
    "EDF": edf_policy,
    "Priority Aware": priority_aware_policy,
}

metrics_rows = []
results = {}

for i, (name, policy_fn) in enumerate(POLICIES.items()):
    print("Running:", name)

    env = FieldVisionStage51Sim(
        trace,
        n_uavs=4,
        dev_cfg=DEV,
        policy_seed=100 + i,
    )

    metrics = env.run(policy_fn)
    metrics["policy"] = name

    metrics_rows.append(metrics)
    results[name] = env.task_results_dataframe()

metrics_df = pd.DataFrame(metrics_rows)

display(metrics_df)

Running: EDF
Running: Priority Aware


,tasks,completion_rate,sla_success_rate,latency_p50_s,latency_p95_s,network_utilization,local_compute_utilization,edge_compute_utilization,cloud_compute_utilization,policy
0,187,1.0,0.802139,0.445988,12.333552,0.531745,0.244685,0.006148,0.023463,EDF
1,187,1.0,0.786096,0.445988,12.333552,0.532922,0.244481,0.006037,0.023556,Priority Aware


## 6. Verify that the refined policies no longer collapse to identical behavior

In [12]:
edf = results["EDF"][
    ["task_id", "target", "network_tier", "completion_time_s"]
].copy()

prio = results["Priority Aware"][
    ["task_id", "target", "network_tier", "completion_time_s"]
].copy()

merged = edf.merge(
    prio,
    on="task_id",
    suffixes=("_edf", "_prio")
)

different_target = (
    merged["target_edf"].fillna("NONE")
    != merged["target_prio"].fillna("NONE")
)

different_tier = (
    merged["network_tier_edf"].fillna("NONE")
    != merged["network_tier_prio"].fillna("NONE")
)

different_completion = (
    (
        merged["completion_time_s_edf"]
        - merged["completion_time_s_prio"]
    ).abs() > 1e-9
).fillna(False)

print("Tasks with different target:", int(different_target.sum()))
print("Tasks with different network tier:", int(different_tier.sum()))
print("Tasks with different completion time:", int(different_completion.sum()))

if (
    different_target.sum()
    + different_tier.sum()
    + different_completion.sum()
    == 0
):
    print(
        "WARNING: policies still behave identically under this trace. "
        "Increase contention or queue depth before RL."
    )
else:
    print("PASS: the two policies now create different system behavior.")

Tasks with different target: 2
Tasks with different network tier: 2
Tasks with different completion time: 16
PASS: the two policies now create different system behavior.


## 7. SLA success by criticality and urgency

In [13]:
rows = []

for policy_name, df in results.items():
    grouped = (
        df.groupby(["criticality", "urgency"])["sla_success"]
        .agg(["count", "mean"])
        .reset_index()
    )

    grouped["policy"] = policy_name
    rows.append(grouped)

sla_grid = pd.concat(rows, ignore_index=True)

display(
    sla_grid.rename(
        columns={"mean": "sla_success_rate"}
    )
)

,criticality,urgency,count,sla_success_rate,policy
0,high,normal,22,0.954545,EDF
1,high,relaxed,11,0.909091,EDF
2,high,urgent,12,0.750000,EDF
3,low,normal,15,0.866667,EDF
4,low,relaxed,9,0.888889,EDF
5,low,urgent,8,0.625000,EDF
6,normal,normal,67,0.746269,EDF
7,normal,relaxed,17,1.000000,EDF
8,normal,urgent,26,0.653846,EDF
9,high,normal,22,0.954545,Priority Aware


## 8. Save refined trace and validation results

In [14]:
refined_trace_path = SIM_DIR / "fieldvision_task_trace_v2.csv"
metrics_path = RESULTS_DIR / "stage5_1_policy_metrics.csv"
sla_path = RESULTS_DIR / "stage5_1_sla_by_criticality_urgency.csv"

trace.to_csv(refined_trace_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

sla_grid.rename(
    columns={"mean": "sla_success_rate"}
).to_csv(
    sla_path,
    index=False
)

for policy_name, df in results.items():
    safe = policy_name.lower().replace(" ", "_")
    df.to_csv(
        RESULTS_DIR / f"stage5_1_tasks_{safe}.csv",
        index=False
    )

config = {
    "seed": SEED,
    "reference_bandwidth_mbps": REFERENCE_BANDWIDTH_MBPS,
    "criticality": CRITICALITY,
    "urgency": URGENCY,
    "note": (
        "Criticality and deadline urgency are sampled independently. "
        "Both remain experimental systems classes until validated "
        "against domain/service requirements."
    ),
}

with open(
    RESULTS_DIR / "stage5_1_refinement_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print("Saved refined task trace:", refined_trace_path)
print("Saved Stage 5.1 results:", RESULTS_DIR)

Saved refined task trace: /Users/bwbgv/Downloads/NOMS_Fieldvision/simulator_inputs/fieldvision_task_trace_v2.csv
Saved Stage 5.1 results: /Users/bwbgv/Downloads/NOMS_Fieldvision/stage5_1_results


# What comes next

If all five controlled validation tests pass and EDF/Priority-Aware no longer collapse to the same behavior, the simulator architecture is ready for the next realism step.

## Stage 6A — Real compute-tier profiling

Re-run the Stage 3 profiler on available hardware representing:
- UAV/onboard compute;
- edge compute;
- cloud/server compute.

This replaces the temporary local/edge/cloud multipliers with measured profiles.

## Stage 6B — Trace-driven wireless model

Replace the development random-walk wireless model with real or realistic network traces.

## Stage 7 — MARL environment

Only then wrap the validated simulator into the MAPPO observation/action/reward interface.